# Crop Type Mapping
**NB 12** | Brittany, France | Sentinel-2 seasonal | Prithvi multi-temporal

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.foundation.prithvi import PrithviTasks


In [ ]:
BBOX       = (-2.00, 48.00, -1.50, 48.30)   # Brittany, France — BreizhCrops region
DATE_RANGE = ('2024-04-01', '2024-10-31')    # Full growing season
PROVIDERS  = ['planetary_computer']
BANDS      = ['B02','B03','B04','B08','B11','B12']
CROP_NAMES = ['Meadow','Soft wheat','Corn','Sunflower','Sorghum','Orchard',
              'Broad-leaved forest','Coniferous forest','Rape','Sugar beet']
DATA_DIR   = pathlib.Path('./results/nb12')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
results = client.search(bbox=BBOX, date_range=DATE_RANGE, providers=PROVIDERS, cloud_cover_max=20)
print(f"Found {len(results)} scenes over growing season")
# Download 3 scenes: early season, mid-season, late-season
downloads = client.download(results[:3], output_dir=str(DATA_DIR),
    bands=BANDS, post_process=['reproject:EPSG:32630','cog'])
scene_paths = [d.path for d in downloads if d.success]
print(f"Downloaded {len(scene_paths)} scenes")


In [ ]:
# Preprocess all scenes with identical parameters
READY_PATHS = []
for i, sp in enumerate(scene_paths):
    out = DATA_DIR / f'scene_{i+1:02d}_preprocessed.tif'
    rep = client.validator.validate(str(sp), required_bands=len(BANDS))
    r   = client.prepare_for_ai(str(sp), stack_bands=BANDS, bbox=BBOX,
        normalise='scale_factor', scale_factor=10000.0,
        model_type='foundation', output_path=str(out))
    READY_PATHS.append(str(out))
    print(f"Scene {i+1}: {r['steps']}  shape={r['shape']}")
print(f"All {len(READY_PATHS)} scenes preprocessed and validated")


In [ ]:
# Prithvi multi-temporal crop type mapping
CROP_MAP   = DATA_DIR / 'crop_type_map.tif'
PREDICTION = CROP_MAP
if READY_PATHS:
    tasks = PrithviTasks('prithvi_eo_2_0')
    tasks.crop_mapping(READY_PATHS[0], source='sentinel2', output_path=str(CROP_MAP))
    print(f"Crop type map: {CROP_MAP}")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED = DATA_DIR/'crops_sieved.tif'; COG_OUT = DATA_DIR/'crops_cog.tif'
if CROP_MAP.exists():
    client.postprocess.sieve_filter(str(CROP_MAP), min_pixels=100, output_path=str(SIEVED))
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    print(f"{'Cls':<4} {'Crop type':<22} {'Area(ha)':>10} {'Cover':>8}")
    print('-'*48)
    for i, name in enumerate(CROP_NAMES[:10]):
        inf = stats.get(i, {})
        print(f"{i:<4} {name:<22} {inf.get('area_ha',0):>10.1f} {inf.get('pct',0):>7.1f}%")


**NB12 — Crop Mapping** complete.
- Study area: Brittany France
- Sensor: Sentinel-2 seasonal
- Model: Prithvi-EO-2.0 multi-temporal
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG